In [2]:
%load_ext autoreload
%autoreload 2

import functools
import itertools
from math import pi
import os
import random


os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize
import scipy.stats.qmc
import tqdm.contrib.itertools
import xarray as xr

from fluxoniumcr import DATA_DIR
from fluxoniumcr.plotting.cookbook import imshow_heatmap
from fluxoniumcr.qubits.fluxonium import Fluxonium
from fluxoniumcr.spectrum.autodiff import FunctionSum, SampledFunction
from fluxoniumcr.spectrum.square_spectrum import SquareSpectrum
from fluxoniumcr.spectrum.esd_oracle import ESDOracle
from fluxoniumcr.utils import load_arguments

In [3]:
RAMP_DURATION = 50.0
RAMP_SHAPE = 'planck'

In [4]:
EJ_0 = 4.0 * (2*pi)
EC_0 = 1.2 * (2*pi)
EL_0 = 0.4 * (2*pi)
deltap = 0.8

parent_path = (
    DATA_DIR
    /"control_spectator_collision"
    / f"EJ={EJ_0/(2*pi):.2f},EC={EC_0/(2*pi):.2f},EL={EL_0/(2*pi):.2f},deltap={deltap}"
)
parent_path.mkdir(parents=True, exist_ok=True)

q0 = Fluxonium(
    EJ=EJ_0,
    EC=EC_0,
    EL=EL_0,
    dim=16,
    cutoff=128,
)
q0_evals = q0.eigenvalues

esd_dataset = xr.load_dataset(
    DATA_DIR/"charge_operator"
    / f"EJ={EJ_0/(2*pi):.2f},EC={EC_0/(2*pi):.2f},EL={EL_0/(2*pi):.2f}"
    / f"esd.hdf5"
).sel(deltap=deltap, method='nearest')

esd_oracle = ESDOracle(esd_dataset)

assert esd_dataset.attrs['ramp_shape'] == RAMP_SHAPE
assert esd_dataset.attrs['ramp_duration'] == RAMP_DURATION

# Functions

In [15]:
_random_sampler = scipy.stats.qmc.Sobol(d=3, scramble=True)

def generate_random_parameters(num):
    # Over sample since we will mask some points later.
    u, v, w = _random_sampler.random(2**int(np.ceil(np.log2(num))+2)).T

    EC_array = (1.0 + 0.5*u) * 2*pi
    EL_array = (0.1 + 1.9*v) * 2*pi
    EJ_array = (2 + 8*w) * 2*pi
    
    ratio = EL_array/EJ_array
    
    mask = (ratio >= 0.1) & (ratio <= 0.5)
    
    EJ_array = EJ_array[mask][:num]
    EC_array = EC_array[mask][:num]
    EL_array = EL_array[mask][:num]
    
    return EJ_array, EC_array, EL_array


def create_dataset(
        spectator_final_state,
        spectator_initial_state,
        control_final_state,
        control_initial_state,
        harmonic,
        delta_frequency,
        JC_values,
        num_points,
        bin_edges=None,
        max_iterations=1000,
):  
    dataset = xr.Dataset(attrs=dict(
        spectator_final_state=spectator_final_state,
        spectator_initial_state=spectator_initial_state,
        control_final_state=control_final_state,
        control_initial_state=control_initial_state,
        harmonic=harmonic,
        ramp_duration=RAMP_DURATION,
        ramp_shape=RAMP_SHAPE,
    ))

    JC_coord = xr.DataArray(
        JC_values,
        dims=['JC'],
    )
    index_coord = xr.DataArray(
        np.arange(num_points),
        dims=['index'],
    )
    
    truncated_dims = (8, 4)
    
    control_bra_coord = xr.DataArray(
        np.arange(truncated_dims[0]),
        dims=['control_bra'],
        attrs=dict(
            long_name="Control final state"
        )
    )
    spectator_bra_coord = xr.DataArray(
        np.arange(truncated_dims[1]),
        dims=['spectator_bra'],
        attrs=dict(
            long_name="Spectator final state"
        )
    )
    control_ket_coord = xr.DataArray(
        np.arange(2),
        dims=['control_ket'],
        attrs=dict(
            long_name="Control initial qubit state"
        )
    )
    spectator_ket_coord = xr.DataArray(
        np.arange(2),
        dims=['spectator_ket'],
        attrs=dict(
            long_name="Spectator initial qubit state"
        )
    )
 
    dataset['transition_probability'] = xr.DataArray(
        np.nan,
        coords=[
            JC_coord,
            index_coord,
            control_bra_coord,
            spectator_bra_coord,
            control_ket_coord,
            spectator_ket_coord,
        ]
    )
    dataset['model_probability'] = xr.DataArray(
        np.nan,
        coords=[JC_coord, index_coord],
        attrs=dict(
            long_name="Model probability"
        )
    )

    keep_params = {}
    
    for _ in range(max_iterations):
        EJ_values, EC_values, EL_values = generate_random_parameters(num_points)
        
        fx = Fluxonium(
            EJ=EJ_values,
            EC=EC_values,
            EL=EL_values,
            flux=0.5,
            dim=3,
            # Spectrum doesn't need to be very accurate.
            cutoff=32,
        )
        fx_evals = fx.eigenvalues
        unwanted_freqs = fx_evals[..., spectator_final_state] - fx_evals[..., spectator_initial_state]
        
        if control_final_state == control_initial_state:
            drive_freqs = unwanted_freqs/abs(harmonic)
        else:
            # Try and determine the dressed resonance frequency iteratively.
            
            drive_freqs = 1/abs(harmonic) * (
                unwanted_freqs
                + q0_evals[control_final_state]
                - q0_evals[control_initial_state]
            )
            
            # XXX: esd_dataset harmonics are calculated with glide reflection symmetry
            # so all harmonics are odd.
            if (
                    (control_initial_state == 0 and control_final_state == 1)
            ):
                assert harmonic % 2 == 0
                modified_harmonic = harmonic + 1
            elif (
                    (control_initial_state == 1 and control_final_state == 0)
                    or (control_initial_state == 1 and control_final_state == 2)
            ):
                assert harmonic % 2 == 0
                modified_harmonic = harmonic - 1
            else:
                assert harmonic % 2 == 1
                modified_harmonic = harmonic
                

            for _ in range(100):
                pole = (
                        esd_dataset.sel(
                            harmonic=modified_harmonic,
                            bra=control_final_state,
                            ket=control_initial_state,
                        )
                        .pole
                        .interp(drive_frequency=drive_freqs)
                        .data
                )
                delta_freqs = unwanted_freqs + pole
                mask = np.isfinite(delta_freqs)
                drive_freqs[mask] += delta_freqs[mask]/abs(harmonic)
                if abs(delta_freqs[mask]).max() < 1e-3 * 2*pi:
                    break
        
        if bin_edges is None:
            break

        # Keep only drive frequencies evenly spaced between bin_edges.
        # Throw away frequencies lower or higher.
        indices = np.digitize(
            drive_freqs,
            [-np.inf, *bin_edges, np.inf],
        )
        
        for i in range(1, len(bin_edges)):
            params_list = keep_params.setdefault(i, [])
            mask = indices == (i+1)
            new_params = list(zip(
                EJ_values[mask],
                EC_values[mask],
                EL_values[mask],
                drive_freqs[mask],
            ))
            params_list.extend(new_params)

        for i in range(1, len(bin_edges)):
            if i not in keep_params:
                break
            params_list = keep_params[i]
            if len(params_list) < np.ceil(num_points/(len(bin_edges) - 1)):
                # Log progress.
                print(i, bin_edges[i-1]/(2*pi), len(params_list))
                break
        else:
            # Done
            total_params_list = list(itertools.chain.from_iterable(
                params_list[:int(np.ceil(num_points/(len(bin_edges) - 1)))]
                for params_list in keep_params.values()
            ))
            random.shuffle(total_params_list)
            total_params_list = total_params_list[:num_points]
            EJ_values, EC_values, EL_values, drive_freqs = map(list, zip(*total_params_list))
            break
    else:
        raise ValueError(f"Sampling exceeded {max_iterations} iterations.")
    
    dataset['EJ'] = xr.DataArray(
        EJ_values,
        coords=[index_coord],
        attrs=dict(
            unit="Grad/s"
        )   
    )
    dataset['EC'] = xr.DataArray(
        EC_values,
        coords=[index_coord],
        attrs=dict(
            unit="Grad/s"
        )   
    )
    dataset['EL'] = xr.DataArray(
        EL_values,
        coords=[index_coord],
        attrs=dict(
            unit="Grad/s"
        )   
    )
    
    sampler = scipy.stats.qmc.Sobol(d=1, scramble=True)
    random_delta_freqs = delta_frequency/abs(harmonic) * (2*sampler.random(len(drive_freqs)).ravel() - 1)
   
    dataset['drive_frequency'] = xr.DataArray(
        drive_freqs + random_delta_freqs,
        coords=[index_coord],
        attrs=dict(
            long_name="Drive frequency",
            unit="Grad/s",
        )
    )
    dataset['transition_frequency'] = xr.DataArray(
        np.nan,
        coords=[index_coord],
        attrs=dict(
            long_name="Spectator qubit unwanted transition frequency",
            unit="Grad/s",
        )
    )
    dataset['delta_frequency'] = xr.DataArray(
        np.nan,
        coords=[index_coord],
        attrs=dict(
            long_name="Detuning from dressed resonance",
            unit="Grad/s",
        )
    )
    dataset['drive_amplitude'] = xr.DataArray(
        np.nan,
        coords=[index_coord],
        attrs=dict(
            long_name="Drive amplitude",
            unit="Grad/s",
        )
    )
    
    return dataset

In [17]:
from multiprocessing.pool import Pool

from fluxoniumcr.qubits.product_basis import DressedProductBasis
from fluxoniumcr.simulation.root import create_two_coupled_qubits
from fluxoniumcr.simulation.cnot_solver import (
    CNOTSolver,
    CNOTParameters,
    SmoothSquareParameters,
)
from fluxoniumcr.simulation.signals import planck_taper_signal, cosine_taper_signal


def doit(
        params,
        JC,
        harmonic,
        spectator_final_state,
        spectator_initial_state,
        control_final_state,
        control_initial_state,
        truncated_dims,
):
    EJ_1, EC_1, EL_1, drive_freq = params
    
    q1 = Fluxonium(
        EJ=EJ_1,
        EC=EC_1,
        EL=EL_1,
        dim=16,
        cutoff=128,
    )
    q1_evals = q1.eigenvalues
    n1_op = q1.get_operator('charge')
    
    unwanted_freq = q1_evals[spectator_final_state] - q1_evals[spectator_initial_state]
    
    try:
        amplitude = esd_oracle.interpolate(drive_freq).drive_amplitude
    except ValueError:
        amplitude = np.nan
    
    if np.isnan(amplitude):
        delta_freq = np.nan
        model_prob = np.nan
        return (
            delta_freq,
            unwanted_freq,
            amplitude,
            np.full((*truncated_dims, 2, 2), np.nan),
            delta_freq,
        )

    if control_final_state == control_initial_state:
        spec0 = esd_oracle.interpolate(drive_freq).get_matrix_element(
            harmonic=harmonic,
            bra=0,
            ket=0,
        )
        spec1 = esd_oracle.interpolate(drive_freq).get_matrix_element(
            harmonic=harmonic,
            bra=1,
            ket=1,
        )
        spec = FunctionSum([spec0, spec1])
        delta_freq = unwanted_freq + harmonic * drive_freq
    else:
        # XXX: esd_dataset harmonics are calculated with glide reflection symmetry
        # so all harmonics are odd.
        if control_initial_state == 0 and control_final_state == 1:
            assert harmonic % 2 == 0
            modified_harmonic = harmonic + 1
        elif (
                (control_initial_state == 1 and control_final_state == 0)
                or (control_initial_state == 1 and control_final_state == 2)
        ):
            assert harmonic % 2 == 0
            modified_harmonic = harmonic - 1
        else:
            assert harmonic % 2 == 1
            modified_harmonic = harmonic
            
        
        spec = esd_oracle.interpolate(drive_freq).get_matrix_element(
            harmonic=modified_harmonic,
            bra=control_final_state,
            ket=control_initial_state,
        )
        delta_freq = unwanted_freq + spec.pole

    model_prob = 0.25 * JC**2 * (
        abs(n1_op[spectator_final_state, spectator_initial_state])**2
        * spec(-unwanted_freq)
    )
    
    if RAMP_SHAPE == 'planck':
        pulse_factory = planck_taper_signal
    elif RAMP_SHAPE == 'cosine':
        pulse_factory = cosine_taper_signal
    else:
        raise ValueError(RAMP_SHAPE)
    
    gate_parameters = CNOTParameters(
        pulse_parameters=SmoothSquareParameters(
            total_duration=0.0,  # Irrelevant parameter.
            ramp_duration=RAMP_DURATION,
            amplitude=amplitude,
            carrier_freq=drive_freq,
        ),
        pulse_factory=pulse_factory,
    )
    
    root = create_two_coupled_qubits(
        q0,
        q1,
        JC,
        truncated_dims=truncated_dims,
    )
    basis = root.get(DressedProductBasis)
    cnot_solver = root.get(CNOTSolver)
    
    probs = cnot_solver.calculate_transition_probabilities(gate_parameters)
    
    transition_probability = probs[np.ix_(
        [
            basis.flat_index((i, j))
            for i in range(truncated_dims[0])
            for j in range(truncated_dims[1])
        ],
        [
            basis.flat_index((i, j))
            for i in range(2)
            for j in range(2)
        ],
    )]
    transition_probability = transition_probability.reshape(
        *truncated_dims, 2, 2,
    )
    
    return (
        delta_freq,
        unwanted_freq,
        amplitude,
        transition_probability,
        model_prob,
    )


def populate_dataset(dataset, num_processes=8):
    harmonic = dataset.attrs['harmonic']
    spectator_final_state = dataset.attrs['spectator_final_state']
    spectator_initial_state = dataset.attrs['spectator_initial_state']
    control_final_state = dataset.attrs['control_final_state']
    control_initial_state = dataset.attrs['control_initial_state']
    truncated_dims = (
        len(dataset.control_bra),
        len(dataset.spectator_bra),
    )
    
    with Pool(processes=num_processes) as pool:
        for JC in dataset.JC.data:
            print(f"{JC/(2*pi)=!s}")
            ds_outer = dataset.loc[dict(JC=JC)]
            for index, return_values in tqdm.contrib.tzip(
                    dataset.index.data,
                    pool.imap(
                        functools.partial(
                            doit,
                            JC=JC,
                            harmonic=harmonic,
                            spectator_final_state=spectator_final_state,
                            spectator_initial_state=spectator_initial_state,
                            control_final_state=control_final_state,
                            control_initial_state=control_initial_state,
                            truncated_dims=truncated_dims,
                        ),
                        zip(
                            dataset.EJ.data,
                            dataset.EC.data,
                            dataset.EL.data,
                            dataset.drive_frequency.data,
                        )
                    ),
            ):
                (
                        delta_freq,
                        unwanted_freq,
                        amplitude,
                        transition_probability,
                        model_prob,
                ) = return_values

                ds = ds_outer.loc[dict(index=index)]
                ds.delta_frequency[()] = delta_freq
                ds.transition_frequency[()] = unwanted_freq
                ds.drive_amplitude[()] = amplitude
                ds.transition_probability[()] = transition_probability
                ds.model_probability[()] = model_prob

In [7]:
def save_dataset(dataset):
    dataset_filename = (
        str(dataset.attrs['control_initial_state'])
        + str(dataset.attrs['spectator_initial_state'])
        + 'to'
        + str(dataset.attrs['control_final_state'])
        + str(dataset.attrs['spectator_final_state'])
        + '_harmonic='
        + str(dataset.attrs['harmonic'])
        + '.hdf5'
    )
    print(f"Saving to '{dataset_filename}'")
    dataset.to_netcdf(parent_path/dataset_filename)

# Calculations

In [10]:
dataset = create_dataset(
    harmonic=-4,
    spectator_final_state=2,
    spectator_initial_state=1,
    control_final_state=1,
    control_initial_state=0,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**17,
    bin_edges=np.linspace(0.80, 1.59, 80) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.8 691
1 0.8 1369
JC/(2*pi)=0.1


  0%|          | 0/131072 [00:00<?, ?it/s]

Saving to '01to12_harmonic=-4.hdf5'


In [9]:
dataset = create_dataset(
    harmonic=-4,
    spectator_final_state=2,
    spectator_initial_state=1,
    control_final_state=0,
    control_initial_state=1,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**17,
    bin_edges=np.linspace(0.80, 1.59, 80) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.8 1393
JC/(2*pi)=0.1


  0%|          | 0/131072 [00:00<?, ?it/s]

Saving to '11to02_harmonic=-4.hdf5'


In [20]:
dataset = create_dataset(
    harmonic=-3,
    spectator_final_state=0,
    spectator_initial_state=1,
    control_final_state=2,
    control_initial_state=0,
    delta_frequency=8e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.82, 1.31, 50) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.8199999999999998 280
1 0.8199999999999998 565
1 0.8199999999999998 838
1 0.8199999999999998 1124
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '01to20_harmonic=-3.hdf5'


In [18]:
dataset = create_dataset(
    harmonic=-2,
    spectator_final_state=0,
    spectator_initial_state=1,
    control_final_state=2,
    control_initial_state=1,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.92, 1.52, 61) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.92 30
1 0.92 59
1 0.92 88
1 0.92 113
1 0.92 141
1 0.92 168
1 0.92 198
1 0.92 238
1 0.92 272
1 0.92 312
1 0.92 345
1 0.92 382
1 0.92 412
1 0.92 443
1 0.92 479
1 0.92 507
1 0.92 541
1 0.92 563
1 0.92 599
1 0.92 638
1 0.92 665
1 0.92 704
1 0.92 737
1 0.92 763
1 0.92 793
1 0.92 822
1 0.92 858
1 0.92 892
1 0.92 928
1 0.92 961
1 0.92 993
1 0.92 1031
1 0.92 1062
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '11to20_harmonic=-2.hdf5'


In [8]:
dataset = create_dataset(
    harmonic=-1,
    spectator_final_state=1,
    spectator_initial_state=0,
    control_final_state=0,
    control_initial_state=0,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.22, 1.52, 131) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

25 0.45999999999999996 484
73 0.94 499
109 1.3 470
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '00to01_harmonic=-1.hdf5'


In [35]:
dataset = create_dataset(
    harmonic=-2,
    spectator_final_state=1,
    spectator_initial_state=0,
    control_final_state=1,
    control_initial_state=0,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.25, 1.05, 81) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

21 0.45 793
46 0.7 799
66 0.9 806
73 0.97 807
78 1.02 780
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '00to11_harmonic=-2.hdf5'


In [19]:
dataset = create_dataset(
    harmonic=-2,
    spectator_final_state=1,
    spectator_initial_state=0,
    control_final_state=0,
    control_initial_state=1,
    delta_frequency=20e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.25, 0.75, 51) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.25 69
1 0.25 134
1 0.25 193
1 0.25 249
1 0.25 304
1 0.25 368
1 0.25 431
1 0.25 499
1 0.25 574
1 0.25 641
1 0.25 692
1 0.25 757
1 0.25 824
1 0.25 887
1 0.25 937
1 0.25 1001
1 0.25 1064
1 0.25 1129
1 0.25 1186
1 0.25 1249
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '10to01_harmonic=-2.hdf5'


In [9]:
dataset = create_dataset(
    harmonic=-3,
    spectator_final_state=1,
    spectator_initial_state=0,
    control_final_state=0,
    control_initial_state=0,
    delta_frequency=8e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**16,
    bin_edges=np.linspace(0.25, 0.65, 41) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.25 967
5 0.29 1581
19 0.43 1565
29 0.53 1573
32 0.56 1516
35 0.5900000000000001 1540
37 0.61 1449
38 0.62 1584
39 0.63 1564
JC/(2*pi)=0.1


  0%|          | 0/65536 [00:00<?, ?it/s]

Saving to '00to01_harmonic=-3.hdf5'


In [8]:
dataset = create_dataset(
    harmonic=-3,
    spectator_final_state=2,
    spectator_initial_state=1,
    control_final_state=0,
    control_initial_state=0,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**17,
    bin_edges=np.linspace(0.80, 1.59, 80) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)

1 0.8 8
1 0.8 16
1 0.8 22
1 0.8 32
1 0.8 37
1 0.8 43
1 0.8 54
1 0.8 61
1 0.8 69
1 0.8 78
1 0.8 84
1 0.8 90
1 0.8 98
1 0.8 105
1 0.8 112
1 0.8 119
1 0.8 125
1 0.8 131
1 0.8 138
1 0.8 147
1 0.8 156
1 0.8 164
1 0.8 170
1 0.8 178
1 0.8 186
1 0.8 192
1 0.8 204
1 0.8 211
1 0.8 219
1 0.8 229
1 0.8 235
1 0.8 243
1 0.8 248
1 0.8 256
1 0.8 265
1 0.8 273
1 0.8 281
1 0.8 289
1 0.8 295
1 0.8 303
1 0.8 309
1 0.8 316
1 0.8 322
1 0.8 331
1 0.8 340
1 0.8 347
1 0.8 354
1 0.8 359
1 0.8 371
1 0.8 377
1 0.8 383
1 0.8 387
1 0.8 393
1 0.8 403
1 0.8 410
1 0.8 419
1 0.8 426
1 0.8 432
1 0.8 441
1 0.8 449
1 0.8 456
1 0.8 460
1 0.8 468
1 0.8 477
1 0.8 486
1 0.8 493
1 0.8 501
1 0.8 506
1 0.8 514
1 0.8 522
1 0.8 529
1 0.8 538
1 0.8 545
1 0.8 554
1 0.8 560
1 0.8 568
1 0.8 574
1 0.8 580
1 0.8 589
1 0.8 596
1 0.8 601
1 0.8 607
1 0.8 615
1 0.8 622
1 0.8 629
1 0.8 635
1 0.8 643
1 0.8 650
1 0.8 655
1 0.8 663
1 0.8 671
1 0.8 680
1 0.8 693
1 0.8 699
1 0.8 706
1 0.8 711
1 0.8 720
1 0.8 726
1 0.8 732
1 0.8 739
1 0.8 748
1 0.

  0%|          | 0/131072 [00:00<?, ?it/s]

Saving to '01to02_harmonic=-3.hdf5'


In [ ]:
dataset = create_dataset(
    harmonic=-2,
    spectator_final_state=2,
    spectator_initial_state=1,
    control_final_state=0,
    control_initial_state=1,
    delta_frequency=80e-3 * 2*pi,
    JC_values=[100e-3 * 2*pi],
    num_points=2**13,
    bin_edges=np.linspace(1.1, 1.6, 51) * 2*pi,
)
populate_dataset(dataset, num_processes=10)
save_dataset(dataset)